# AD versus FTD differential validation with repeated nested cross-validation

This diagnostic notebook investigates why the original AD-versus-FTD out-of-fold ROC AUC was below 0.50. It verifies class orientation, quantifies split-to-split variability, compares matched selected dimensionalities, evaluates a three-class model, and uses label permutation to distinguish chance variation from systematic inversion.

## tl;dr

- Repeated nested validation compared PSD, HCFIM, and PSD+HCFIM at the same retained dimensionality, $k=30$.
- Mean AD-versus-FTD ROC AUCs were **0.542** for PSD, **0.458** for HCFIM, and **0.445** for PSD+HCFIM.
- The paired PSD+HCFIM-minus-PSD AUC difference averaged **-0.098**, with a 2.5th–97.5th across-repetition percentile range of **[-0.248, -0.013]**.
- The class-orientation audit passed: AD was class 0, FTD was class 1, and ROC AUC used the FTD probability column.
- With all fixed pipelines matched at $k=30$, lower-tail permutation $p$ values were **0.393** for PSD, **0.058** for HCFIM, and **0.036** for PSD+HCFIM; two-sided values around AUC 0.50 were **0.704**, **0.104**, and **0.065**. These are conditional diagnostics and do not correct the broader model search.
- The repeated three-class macro OvR AUC averaged **0.659** with an across-repetition percentile range of **[0.631, 0.685]**. These results do not support differential-diagnostic use and must not be rescued by post hoc probability inversion.


## Context & Methods

### Key assumptions

- Alzheimer disease is encoded as class 0 and frontotemporal dementia as class 1; binary ROC AUC uses the probability assigned to class 1.
- All imputation, scaling, univariate selection, and model fitting occur inside the relevant training fold.
- Repeated nested cross-validation uses 10 independent five-fold outer partitions and three-fold inner tuning.
- PSD, HCFIM, and PSD+HCFIM are compared with the same outer partitions and with both 21 and 30 retained features.
- The 999-label permutation analysis refits a fixed Extra Trees pipeline, including fold-local preprocessing and feature selection. It is a conditional diagnostic and does not correct the broader model search.
- Percentile ranges across the 10 repetitions describe split-to-split variability; they are not independent-cohort confidence intervals.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import sys
import time
import warnings
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import Markdown, display
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 17536
N_REPEATS = 10
OUTER_FOLDS = 5
INNER_FOLDS = 3
N_PERMUTATIONS = 999
OBSERVED_TREES = 300
PERMUTATION_TREES = 100
PERMUTATION_JOBS = min(4, os.cpu_count() or 1)

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT_ROOT = resolve_project_root()
INPUT_PATH = PROJECT_ROOT / "results" / "subject_features_full.csv"
OUT_DIR = Path(os.environ.get(
    "HCFIM_NOTEBOOK07_OUT",
    str(PROJECT_ROOT / "results" / "ad_ftd_differential_validation"),
))
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
COLORS = {
    "PSD": "#315B7D",
    "HCFIM": "#D18B32",
    "PSD+HCFIM": "#8A5A9E",
    "chance": "#555555",
}

print(f"Python: {sys.version.split()[0]}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Input: {INPUT_PATH}")
print(f"Output: {OUT_DIR}")

## Data

### 1. Load and validate the participant-level matrix

In [ ]:
subject_features = pd.read_csv(INPUT_PATH)
required_columns = {"participant_id", "diagnosis"}
assert required_columns.issubset(subject_features.columns)
assert subject_features["participant_id"].is_unique

binary_frame = subject_features.loc[
    subject_features["diagnosis"].isin(["alzheimer", "frontotemporal_dementia"])
].copy().reset_index(drop=True)
three_class_frame = subject_features.loc[
    subject_features["diagnosis"].isin(["control", "alzheimer", "frontotemporal_dementia"])
].copy().reset_index(drop=True)

psd_columns = [column for column in subject_features if column.startswith("psd__")]
hcfim_columns = [column for column in subject_features if column.startswith("hcfim__")]
family_columns = {
    "PSD": psd_columns,
    "HCFIM": hcfim_columns,
    "PSD+HCFIM": psd_columns + hcfim_columns,
}

assert len(binary_frame) == 59
assert binary_frame["diagnosis"].value_counts().to_dict() == {
    "alzheimer": 36,
    "frontotemporal_dementia": 23,
}
assert len(psd_columns) == 210
assert len(hcfim_columns) == 9450
assert np.isfinite(subject_features[psd_columns + hcfim_columns].to_numpy(float)).all()

data_inventory = pd.DataFrame({
    "item": ["AD participants", "FTD participants", "Controls", "PSD features", "HCFIM features"],
    "value": [36, 23, 29, len(psd_columns), len(hcfim_columns)],
})
display(data_inventory)

### 2. Verify label and probability orientation

In [ ]:
binary_label_map = {"alzheimer": 0, "frontotemporal_dementia": 1}
y_binary = binary_frame["diagnosis"].map(binary_label_map).to_numpy(int)
assert np.array_equal(np.unique(y_binary), np.array([0, 1]))

orientation_audit = pd.DataFrame({
    "diagnosis": ["alzheimer", "frontotemporal_dementia"],
    "encoded_label": [0, 1],
    "AUC_probability_column": ["p_alzheimer (diagnostic only)", "p_frontotemporal_dementia (primary)"],
})
display(orientation_audit)

## Results

### 3. Define the repeated nested validation

In [ ]:
def make_pipeline(k: int, seed: int, n_estimators: int, max_depth=None) -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(f_classif, k=k)),
        ("model", ExtraTreesClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            class_weight="balanced",
            random_state=seed,
            n_jobs=1,
        )),
    ])


def binary_metrics(y_true: np.ndarray, probability: np.ndarray, prediction: np.ndarray) -> dict:
    return {
        "roc_auc": roc_auc_score(y_true, probability),
        "pr_auc": average_precision_score(y_true, probability),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "f1_macro": f1_score(y_true, prediction, average="macro"),
    }


def repeated_nested_binary(frame: pd.DataFrame, family: str, k: int):
    columns = family_columns[family]
    X = frame[columns].to_numpy(float)
    y = frame["diagnosis"].map(binary_label_map).to_numpy(int)
    k = min(k, X.shape[1])

    repeat_rows, fold_rows, oof_rows, selected_rows = [], [], [], []
    for repeat in range(N_REPEATS):
        outer_seed = SEED + repeat * 1000
        outer_cv = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=outer_seed)
        oof_probability = np.full(len(y), np.nan)
        oof_prediction = np.full(len(y), -1)

        for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
            fit_seed = SEED + repeat * 1000 + fold
            inner_cv = StratifiedKFold(
                INNER_FOLDS,
                shuffle=True,
                random_state=SEED + repeat * 1000 + 100 + fold,
            )
            search = GridSearchCV(
                make_pipeline(k, fit_seed, OBSERVED_TREES),
                {"model__max_depth": [2, 3, None]},
                scoring="roc_auc",
                cv=inner_cv,
                n_jobs=1,
                refit=True,
            )
            search.fit(X[train_idx], y[train_idx])
            estimator = search.best_estimator_
            classes = estimator.named_steps["model"].classes_
            assert np.array_equal(classes, np.array([0, 1]))
            positive_index = int(np.flatnonzero(classes == 1)[0])
            fold_probability = estimator.predict_proba(X[test_idx])[:, positive_index]
            fold_prediction = (fold_probability >= 0.50).astype(int)
            oof_probability[test_idx] = fold_probability
            oof_prediction[test_idx] = fold_prediction

            fold_auc = roc_auc_score(y[test_idx], fold_probability)
            fold_rows.append({
                "family": family,
                "k": k,
                "repeat": repeat,
                "fold": fold,
                "outer_seed": outer_seed,
                "best_max_depth": str(search.best_params_["model__max_depth"]),
                "inner_best_auc": search.best_score_,
                "fold_roc_auc": fold_auc,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "classes": "|".join(map(str, classes)),
            })

            support = estimator.named_steps["select"].get_support()
            for feature in np.asarray(columns)[support]:
                selected_rows.append({
                    "family": family,
                    "k": k,
                    "repeat": repeat,
                    "fold": fold,
                    "feature": feature,
                })

        assert np.isfinite(oof_probability).all()
        assert (oof_prediction >= 0).all()
        metrics = binary_metrics(y, oof_probability, oof_prediction)
        repeat_rows.append({"family": family, "k": k, "repeat": repeat, "outer_seed": outer_seed, **metrics})

        for index, row in frame.iterrows():
            oof_rows.append({
                "participant_id": row["participant_id"],
                "diagnosis": row["diagnosis"],
                "y_true": int(y[index]),
                "family": family,
                "k": k,
                "repeat": repeat,
                "outer_seed": outer_seed,
                "probability_ftd": oof_probability[index],
                "probability_ad": 1.0 - oof_probability[index],
                "y_pred": int(oof_prediction[index]),
            })

    return (
        pd.DataFrame(repeat_rows),
        pd.DataFrame(fold_rows),
        pd.DataFrame(oof_rows),
        pd.DataFrame(selected_rows),
    )

### 4. Run matched-dimensionality binary experiments

In [ ]:
binary_start = time.time()
binary_outputs = []
for family in ["PSD", "HCFIM", "PSD+HCFIM"]:
    for k in [21, 30]:
        print(f"Running {family}, k={k} ...")
        binary_outputs.append(repeated_nested_binary(binary_frame, family, k))

repeat_metrics = pd.concat([item[0] for item in binary_outputs], ignore_index=True)
fold_metrics = pd.concat([item[1] for item in binary_outputs], ignore_index=True)
binary_oof = pd.concat([item[2] for item in binary_outputs], ignore_index=True)
selected_features = pd.concat([item[3] for item in binary_outputs], ignore_index=True)

repeat_metrics.to_csv(OUT_DIR / "repeated_binary_metrics.csv", index=False)
fold_metrics.to_csv(OUT_DIR / "repeated_binary_fold_metrics.csv", index=False)
binary_oof.to_csv(OUT_DIR / "repeated_binary_oof_predictions.csv", index=False)
selected_features.to_csv(OUT_DIR / "repeated_binary_selected_features.csv", index=False)

print(f"Binary repeated nested analysis completed in {(time.time() - binary_start) / 60:.1f} min")

### 5. Summarize split-to-split variability and matched comparisons

In [ ]:
def percentile_low(values):
    return np.percentile(values, 2.5)


def percentile_high(values):
    return np.percentile(values, 97.5)


summary_rows = []
for (family, k), group in repeat_metrics.groupby(["family", "k"], sort=False):
    summary_rows.append({
        "family": family,
        "k": k,
        "n_repeats": len(group),
        "auc_mean": group["roc_auc"].mean(),
        "auc_median": group["roc_auc"].median(),
        "auc_sd": group["roc_auc"].std(ddof=1),
        "auc_p2_5": percentile_low(group["roc_auc"]),
        "auc_p97_5": percentile_high(group["roc_auc"]),
        "proportion_auc_gt_0_5": (group["roc_auc"] > 0.5).mean(),
        "proportion_auc_lt_0_5": (group["roc_auc"] < 0.5).mean(),
        "pr_auc_mean": group["pr_auc"].mean(),
        "balanced_accuracy_mean": group["balanced_accuracy"].mean(),
        "f1_macro_mean": group["f1_macro"].mean(),
    })
repeated_summary = pd.DataFrame(summary_rows)
repeated_summary.to_csv(OUT_DIR / "repeated_binary_summary.csv", index=False)
display(repeated_summary.style.format(precision=3))

paired_rows = []
for k in [21, 30]:
    wide = repeat_metrics.loc[repeat_metrics["k"] == k].pivot(
        index="repeat", columns="family", values="roc_auc"
    )
    for comparison, left, right in [
        ("PSD+HCFIM minus PSD", "PSD+HCFIM", "PSD"),
        ("HCFIM minus PSD", "HCFIM", "PSD"),
    ]:
        delta = wide[left] - wide[right]
        paired_rows.append({
            "comparison": comparison,
            "k": k,
            "mean_delta_auc": delta.mean(),
            "median_delta_auc": delta.median(),
            "delta_p2_5": np.percentile(delta, 2.5),
            "delta_p97_5": np.percentile(delta, 97.5),
            "proportion_delta_gt_0": (delta > 0).mean(),
        })
paired_differences = pd.DataFrame(paired_rows)
paired_differences.to_csv(OUT_DIR / "paired_auc_differences.csv", index=False)
display(paired_differences.style.format(precision=3))

### 6. Quantify selected-feature stability

In [ ]:
stability_rows = []
frequency_rows = []
for (family, k), group in selected_features.groupby(["family", "k"], sort=False):
    sets = [set(item["feature"]) for _, item in group.groupby(["repeat", "fold"])]
    jaccards = []
    for left, right in combinations(sets, 2):
        union = left | right
        jaccards.append(len(left & right) / len(union) if union else 1.0)
    stability_rows.append({
        "family": family,
        "k": k,
        "fold_models": len(sets),
        "mean_pairwise_jaccard": np.mean(jaccards),
        "median_pairwise_jaccard": np.median(jaccards),
    })
    counts = group["feature"].value_counts()
    for feature, count in counts.items():
        frequency_rows.append({
            "family": family,
            "k": k,
            "feature": feature,
            "selected_fold_count": count,
            "selected_fold_fraction": count / len(sets),
        })

selection_stability = pd.DataFrame(stability_rows)
selection_frequency = pd.DataFrame(frequency_rows)
selection_stability.to_csv(OUT_DIR / "repeated_selection_stability.csv", index=False)
selection_frequency.to_csv(OUT_DIR / "repeated_selection_frequency.csv", index=False)
display(selection_stability.style.format(precision=3))

### 7. Visualize the repeated binary estimates

In [ ]:
plot_frame = repeat_metrics.copy()
plot_frame["configuration"] = plot_frame["family"] + "\nk=" + plot_frame["k"].astype(str)
order = [f"{family}\nk={k}" for k in [21, 30] for family in ["PSD", "HCFIM", "PSD+HCFIM"]]
palette = {
    f"{family}\nk={k}": COLORS[family]
    for k in [21, 30]
    for family in ["PSD", "HCFIM", "PSD+HCFIM"]
}

fig, ax = plt.subplots(figsize=(10.5, 5.8))
sns.boxplot(
    data=plot_frame,
    x="configuration",
    y="roc_auc",
    order=order,
    palette=palette,
    width=0.55,
    showfliers=False,
    ax=ax,
)
sns.stripplot(
    data=plot_frame,
    x="configuration",
    y="roc_auc",
    order=order,
    color="#222222",
    size=4.5,
    jitter=0.12,
    ax=ax,
)
ax.axhline(0.5, color=COLORS["chance"], linestyle="--", linewidth=1.2, label="Chance AUC")
ax.set(title="AD versus FTD ROC AUC across repeated nested partitions", xlabel="Feature family and retained features", ylabel="Out-of-fold ROC AUC")
ax.set_ylim(0.15, 0.85)
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
fig.savefig(FIG_DIR / "repeated_binary_auc_distributions.png", dpi=220, bbox_inches="tight")
plt.show()

### 8. Build consensus out-of-fold predictions and audit orientation

In [ ]:
consensus_oof = (
    binary_oof.groupby(["participant_id", "diagnosis", "y_true", "family", "k"], as_index=False)
    .agg(
        probability_ftd=("probability_ftd", "mean"),
        probability_ftd_sd=("probability_ftd", "std"),
    )
)
consensus_oof["probability_ad"] = 1.0 - consensus_oof["probability_ftd"]
consensus_oof["y_pred"] = (consensus_oof["probability_ftd"] >= 0.50).astype(int)
consensus_oof.to_csv(OUT_DIR / "consensus_binary_oof_predictions.csv", index=False)

orientation_rows = []
consensus_metric_rows = []
for (family, k), group in consensus_oof.groupby(["family", "k"], sort=False):
    auc_ftd = roc_auc_score(group["y_true"], group["probability_ftd"])
    auc_wrong_probability = roc_auc_score(group["y_true"], group["probability_ad"])
    assert np.isclose(auc_ftd + auc_wrong_probability, 1.0)
    orientation_rows.append({
        "family": family,
        "k": k,
        "AUC_using_p_FTD": auc_ftd,
        "AUC_using_p_AD_with_FTD_labels": auc_wrong_probability,
        "sum": auc_ftd + auc_wrong_probability,
        "estimator_class_order_verified": "0|1 in every fold",
    })
    metrics = binary_metrics(
        group["y_true"].to_numpy(),
        group["probability_ftd"].to_numpy(),
        group["y_pred"].to_numpy(),
    )
    tn, fp, fn, tp = confusion_matrix(group["y_true"], group["y_pred"]).ravel()
    consensus_metric_rows.append({
        "family": family,
        "k": k,
        **metrics,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    })

orientation_results = pd.DataFrame(orientation_rows)
consensus_binary_metrics = pd.DataFrame(consensus_metric_rows)
orientation_results.to_csv(OUT_DIR / "class_orientation_audit.csv", index=False)
consensus_binary_metrics.to_csv(OUT_DIR / "consensus_binary_metrics.csv", index=False)
display(orientation_results.style.format(precision=3))
display(consensus_binary_metrics.style.format(precision=3))

### 9. Plot consensus ROC, precision–recall, and confusion matrices

In [ ]:
primary_configs = [("PSD", 30), ("HCFIM", 30), ("PSD+HCFIM", 30)]

fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.1))
for family, k in primary_configs:
    group = consensus_oof.loc[(consensus_oof["family"] == family) & (consensus_oof["k"] == k)]
    fpr, tpr, _ = roc_curve(group["y_true"], group["probability_ftd"])
    precision, recall, _ = precision_recall_curve(group["y_true"], group["probability_ftd"])
    auc_value = roc_auc_score(group["y_true"], group["probability_ftd"])
    ap_value = average_precision_score(group["y_true"], group["probability_ftd"])
    label = f"{family}, k={k} (AUC={auc_value:.3f})"
    axes[0].plot(fpr, tpr, color=COLORS[family], linewidth=2, label=label)
    axes[1].plot(recall, precision, color=COLORS[family], linewidth=2, label=f"{family}, k={k} (AP={ap_value:.3f})")

axes[0].plot([0, 1], [0, 1], linestyle="--", color=COLORS["chance"], linewidth=1)
axes[0].set(title="Consensus out-of-fold ROC curves", xlabel="False-positive rate", ylabel="True-positive rate", xlim=(0, 1), ylim=(0, 1))
prevalence = y_binary.mean()
axes[1].axhline(prevalence, linestyle="--", color=COLORS["chance"], linewidth=1, label=f"FTD prevalence={prevalence:.3f}")
axes[1].set(title="Consensus out-of-fold precision–recall curves", xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1))
for ax in axes:
    ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "consensus_binary_roc_pr.png", dpi=220, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.8))
for ax, (family, k) in zip(axes, primary_configs):
    group = consensus_oof.loc[(consensus_oof["family"] == family) & (consensus_oof["k"] == k)]
    ConfusionMatrixDisplay.from_predictions(
        group["y_true"],
        group["y_pred"],
        display_labels=["AD", "FTD"],
        normalize="true",
        cmap="Blues",
        colorbar=False,
        values_format=".1%",
        ax=ax,
    )
    ax.set_title(f"{family}, k={k}")
fig.suptitle("Consensus out-of-fold confusion matrices", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "consensus_binary_confusion_matrices.png", dpi=220, bbox_inches="tight")
plt.show()

### 10. Evaluate the three-class model with repeated nested validation

In [ ]:
three_label_order = ["control", "alzheimer", "frontotemporal_dementia"]
three_label_map = {label: index for index, label in enumerate(three_label_order)}
three_columns = family_columns["PSD+HCFIM"]
X_three = three_class_frame[three_columns].to_numpy(float)
y_three = three_class_frame["diagnosis"].map(three_label_map).to_numpy(int)

three_repeat_rows, three_oof_rows = [], []
three_start = time.time()
for repeat in range(N_REPEATS):
    outer_seed = SEED + repeat * 1000
    outer_cv = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=outer_seed)
    probabilities = np.full((len(y_three), 3), np.nan)

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_three, y_three), start=1):
        inner_cv = StratifiedKFold(
            INNER_FOLDS,
            shuffle=True,
            random_state=SEED + repeat * 1000 + 100 + fold,
        )
        search = GridSearchCV(
            make_pipeline(30, SEED + repeat * 1000 + fold, OBSERVED_TREES),
            {"model__max_depth": [2, 3, None]},
            scoring="roc_auc_ovr_weighted",
            cv=inner_cv,
            n_jobs=1,
            refit=True,
        )
        search.fit(X_three[train_idx], y_three[train_idx])
        estimator = search.best_estimator_
        classes = estimator.named_steps["model"].classes_
        assert np.array_equal(classes, np.array([0, 1, 2]))
        probabilities[test_idx] = estimator.predict_proba(X_three[test_idx])

    predictions = np.argmax(probabilities, axis=1)
    macro_auc = roc_auc_score(y_three, probabilities, multi_class="ovr", average="macro")
    three_repeat_rows.append({
        "repeat": repeat,
        "outer_seed": outer_seed,
        "macro_ovr_auc": macro_auc,
        "balanced_accuracy": balanced_accuracy_score(y_three, predictions),
        "f1_macro": f1_score(y_three, predictions, average="macro"),
    })
    for index, row in three_class_frame.iterrows():
        three_oof_rows.append({
            "participant_id": row["participant_id"],
            "diagnosis": row["diagnosis"],
            "y_true": int(y_three[index]),
            "repeat": repeat,
            "p_control": probabilities[index, 0],
            "p_alzheimer": probabilities[index, 1],
            "p_frontotemporal_dementia": probabilities[index, 2],
            "y_pred": int(predictions[index]),
        })

three_repeat_metrics = pd.DataFrame(three_repeat_rows)
three_oof = pd.DataFrame(three_oof_rows)
three_repeat_metrics.to_csv(OUT_DIR / "repeated_three_class_metrics.csv", index=False)
three_oof.to_csv(OUT_DIR / "repeated_three_class_oof_predictions.csv", index=False)
print(f"Three-class repeated nested analysis completed in {(time.time() - three_start) / 60:.1f} min")
display(three_repeat_metrics.describe(percentiles=[0.025, 0.5, 0.975]).T.style.format(precision=3))

In [ ]:
three_consensus = (
    three_oof.groupby(["participant_id", "diagnosis", "y_true"], as_index=False)
    .agg(
        p_control=("p_control", "mean"),
        p_alzheimer=("p_alzheimer", "mean"),
        p_frontotemporal_dementia=("p_frontotemporal_dementia", "mean"),
    )
)
three_probability_columns = ["p_control", "p_alzheimer", "p_frontotemporal_dementia"]
three_consensus["y_pred"] = np.argmax(three_consensus[three_probability_columns].to_numpy(), axis=1)
three_consensus.to_csv(OUT_DIR / "consensus_three_class_oof_predictions.csv", index=False)

three_consensus_probabilities = three_consensus[three_probability_columns].to_numpy()
three_consensus_auc = roc_auc_score(
    three_consensus["y_true"],
    three_consensus_probabilities,
    multi_class="ovr",
    average="macro",
)
one_vs_rest_rows = []
for class_index, label in enumerate(three_label_order):
    binary_target = (three_consensus["y_true"].to_numpy() == class_index).astype(int)
    one_vs_rest_rows.append({
        "class": label,
        "one_vs_rest_auc": roc_auc_score(binary_target, three_consensus_probabilities[:, class_index]),
    })
three_ovr_auc = pd.DataFrame(one_vs_rest_rows)
three_ovr_auc.to_csv(OUT_DIR / "consensus_three_class_ovr_auc.csv", index=False)
display(Markdown(f"Consensus three-class macro OvR AUC: **{three_consensus_auc:.3f}**"))
display(three_ovr_auc.style.format(precision=3))

fig, ax = plt.subplots(figsize=(5.3, 4.6))
ConfusionMatrixDisplay.from_predictions(
    three_consensus["y_true"],
    three_consensus["y_pred"],
    display_labels=["Control", "AD", "FTD"],
    normalize="true",
    cmap="Blues",
    colorbar=False,
    values_format=".1%",
    ax=ax,
)
ax.set_title("Three-class consensus out-of-fold confusion matrix")
fig.tight_layout()
fig.savefig(FIG_DIR / "consensus_three_class_confusion_matrix.png", dpi=220, bbox_inches="tight")
plt.show()

### 11. Run 999-label permutation diagnostics

The permutation test uses a fixed Extra Trees pipeline with fold-local imputation, scaling, ANOVA feature selection, and model fitting. Hyperparameters are not reselected inside an inner loop for each permutation. Consequently, this evaluates whether the specified fixed pipeline departs from its label-null distribution; it does not validate the broader model-search process.

In [ ]:
def fixed_binary_cv_auc(X: np.ndarray, y: np.ndarray, k: int, seed: int) -> float:
    cv = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=SEED)
    probability = np.full(len(y), np.nan)
    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        estimator = make_pipeline(
            min(k, X.shape[1]),
            seed + fold,
            PERMUTATION_TREES,
            max_depth=3,
        )
        estimator.fit(X[train_idx], y[train_idx])
        classes = estimator.named_steps["model"].classes_
        positive_index = int(np.flatnonzero(classes == 1)[0])
        probability[test_idx] = estimator.predict_proba(X[test_idx])[:, positive_index]
    return roc_auc_score(y, probability)


def fixed_three_class_cv_auc(X: np.ndarray, y: np.ndarray, k: int, seed: int) -> float:
    cv = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=SEED)
    probability = np.full((len(y), 3), np.nan)
    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        estimator = make_pipeline(
            min(k, X.shape[1]),
            seed + fold,
            PERMUTATION_TREES,
            max_depth=3,
        )
        estimator.fit(X[train_idx], y[train_idx])
        probability[test_idx] = estimator.predict_proba(X[test_idx])
    return roc_auc_score(y, probability, multi_class="ovr", average="macro")


def permuted_binary_auc(X: np.ndarray, y: np.ndarray, k: int, permutation_index: int) -> float:
    rng = np.random.default_rng(SEED + 100_000 + permutation_index)
    permuted_y = rng.permutation(y)
    return fixed_binary_cv_auc(X, permuted_y, k, SEED + 200_000 + permutation_index * 10)


def permuted_three_class_auc(X: np.ndarray, y: np.ndarray, k: int, permutation_index: int) -> float:
    rng = np.random.default_rng(SEED + 300_000 + permutation_index)
    permuted_y = rng.permutation(y)
    return fixed_three_class_cv_auc(X, permuted_y, k, SEED + 400_000 + permutation_index * 10)


# Only the PSD specification changed from k=21 to matched k=30.
# Reuse the previously executed null distributions for the three unchanged k=30
# specifications and recompute the modified PSD pipeline with the same 999-permutation protocol.
existing_summary_path = OUT_DIR / "permutation_summary.csv"
existing_distribution_path = OUT_DIR / "permutation_distribution.csv"
existing_permutation_summary = pd.read_csv(existing_summary_path)
existing_permutation_distribution = pd.read_csv(existing_distribution_path)

unchanged_keys = {
    ("AD_vs_FTD", "HCFIM", 30),
    ("AD_vs_FTD", "PSD+HCFIM", 30),
    ("Control_vs_AD_vs_FTD", "PSD+HCFIM", 30),
}
unchanged_summary = existing_permutation_summary.loc[
    existing_permutation_summary.apply(
        lambda row: (row["task"], row["family"], int(row["k"])) in unchanged_keys, axis=1
    )
].copy()
unchanged_distribution = existing_permutation_distribution.loc[
    existing_permutation_distribution.apply(
        lambda row: (row["task"], row["family"], int(row["k"])) in unchanged_keys, axis=1
    )
].copy()
assert len(unchanged_summary) == 3
assert len(unchanged_distribution) == 3 * N_PERMUTATIONS

permutation_specs = [
    ("AD_vs_FTD", "PSD", binary_frame[family_columns["PSD"]].to_numpy(float), y_binary, 30, "binary"),
]

permutation_distribution_rows = unchanged_distribution.to_dict("records")
permutation_summary_rows = unchanged_summary.to_dict("records")
permutation_start = time.time()

for task, family, X, y, k, task_type in permutation_specs:
    print(f"Permuting {task}, {family}, k={k} ...")
    if task_type == "binary":
        observed = fixed_binary_cv_auc(X, y, k, SEED + 500_000)
        null_values = Parallel(n_jobs=PERMUTATION_JOBS, prefer="threads")(
            delayed(permuted_binary_auc)(X, y, k, index)
            for index in range(N_PERMUTATIONS)
        )
    else:
        observed = fixed_three_class_cv_auc(X, y, k, SEED + 500_000)
        null_values = Parallel(n_jobs=PERMUTATION_JOBS, prefer="threads")(
            delayed(permuted_three_class_auc)(X, y, k, index)
            for index in range(N_PERMUTATIONS)
        )

    null_values = np.asarray(null_values, dtype=float)
    p_upper = (1 + np.sum(null_values >= observed)) / (N_PERMUTATIONS + 1)
    p_lower = (1 + np.sum(null_values <= observed)) / (N_PERMUTATIONS + 1)
    p_two_sided = (
        1 + np.sum(np.abs(null_values - 0.5) >= abs(observed - 0.5))
    ) / (N_PERMUTATIONS + 1)

    permutation_summary_rows.append({
        "task": task,
        "family": family,
        "k": k,
        "observed_auc": observed,
        "null_mean": null_values.mean(),
        "null_sd": null_values.std(ddof=1),
        "null_p2_5": np.percentile(null_values, 2.5),
        "null_p97_5": np.percentile(null_values, 97.5),
        "p_upper": p_upper,
        "p_lower": p_lower,
        "p_two_sided_around_0_5": p_two_sided,
        "n_permutations": N_PERMUTATIONS,
        "trees": PERMUTATION_TREES,
        "max_depth": 3,
    })
    for index, value in enumerate(null_values):
        permutation_distribution_rows.append({
            "task": task,
            "family": family,
            "k": k,
            "permutation": index,
            "null_auc": value,
        })

permutation_summary = pd.DataFrame(permutation_summary_rows)
permutation_distribution = pd.DataFrame(permutation_distribution_rows)
permutation_summary.to_csv(OUT_DIR / "permutation_summary.csv", index=False)
permutation_distribution.to_csv(OUT_DIR / "permutation_distribution.csv", index=False)
print(f"Matched-k PSD permutation completed in {(time.time() - permutation_start) / 60:.1f} min; three unchanged k=30 distributions were reused from the prior executed audit.")
display(permutation_summary.style.format(precision=4))

### 12. Visualize the permutation diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.3), sharex=True)
for ax, summary_row in zip(axes.flat, permutation_summary.to_dict("records")):
    subset = permutation_distribution.loc[
        (permutation_distribution["task"] == summary_row["task"])
        & (permutation_distribution["family"] == summary_row["family"])
        & (permutation_distribution["k"] == summary_row["k"])
    ]
    ax.hist(subset["null_auc"], bins=28, color="#9FB8CC", edgecolor="white")
    ax.axvline(0.5, color=COLORS["chance"], linestyle="--", linewidth=1.1, label="Chance")
    ax.axvline(summary_row["observed_auc"], color="#B04A5A", linewidth=2.1, label="Observed")
    ax.set(
        title=f"{summary_row['task']}: {summary_row['family']}, k={summary_row['k']}",
        xlabel="ROC AUC under permuted labels",
        ylabel="Permutations",
    )
    ax.text(
        0.02,
        0.96,
        f"Observed={summary_row['observed_auc']:.3f}\n"
        f"p_upper={summary_row['p_upper']:.3f}\n"
        f"p_lower={summary_row['p_lower']:.3f}\n"
        f"p_two-sided={summary_row['p_two_sided_around_0_5']:.3f}",
        transform=ax.transAxes,
        va="top",
        fontsize=9,
    )
    ax.legend(frameon=False, fontsize=8)
fig.suptitle("Fixed-pipeline label-permutation diagnostics", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "label_permutation_diagnostics.png", dpi=220, bbox_inches="tight")
plt.show()

## Takeaways

In [ ]:
primary_summary = repeated_summary.set_index(["family", "k"])
paired_k30 = paired_differences.loc[
    (paired_differences["comparison"] == "PSD+HCFIM minus PSD")
    & (paired_differences["k"] == 30)
].iloc[0]
binary_perm = permutation_summary.loc[permutation_summary["task"] == "AD_vs_FTD"].set_index("family")

takeaway_text = f'''
1. **Class orientation was correct.** Every binary fold exposed estimator classes `0|1`, and ROC AUC used the probability assigned to FTD (class 1). Using the AD probability against FTD labels produced the exact complementary AUC, so the below-chance result is not explained by a probability-column swap.
2. **Performance was partition-sensitive.** Across 10 repeated nested partitions, mean ROC AUC was {primary_summary.loc[("PSD", 30), "auc_mean"]:.3f} for PSD (k=30), {primary_summary.loc[("HCFIM", 30), "auc_mean"]:.3f} for HCFIM (k=30), and {primary_summary.loc[("PSD+HCFIM", 30), "auc_mean"]:.3f} for PSD+HCFIM (k=30).
3. **Matched dimensionality did not establish complementarity.** At k=30, the mean paired AUC difference for PSD+HCFIM minus PSD was {paired_k30["mean_delta_auc"]:+.3f}, with a split-to-split percentile range of [{paired_k30["delta_p2_5"]:+.3f}, {paired_k30["delta_p97_5"]:+.3f}].
4. **Permutation results are diagnostic, not confirmatory.** The fixed-pipeline lower-tail p-values were {binary_perm.loc["PSD", "p_lower"]:.3f} for PSD, {binary_perm.loc["HCFIM", "p_lower"]:.3f} for HCFIM, and {binary_perm.loc["PSD+HCFIM", "p_lower"]:.3f} for PSD+HCFIM. These tests assess systematic inversion relative to the specified pipeline but do not correct the preceding feature/model search.
5. **Clinical interpretation remains unsupported.** The analysis should be reported as an instability and differential-diagnosis failure analysis, not as evidence that post hoc inversion would create a valid classifier.
'''
display(Markdown(takeaway_text))

### Save manifest and execution record

In [ ]:
manifest = {
    "project_root": str(PROJECT_ROOT),
    "input": str(INPUT_PATH),
    "output": str(OUT_DIR),
    "seed": SEED,
    "n_repeats": N_REPEATS,
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "n_permutations": N_PERMUTATIONS,
    "observed_trees": OBSERVED_TREES,
    "permutation_trees": PERMUTATION_TREES,
    "permutation_jobs": PERMUTATION_JOBS,
    "python": sys.version,
    "platform": platform.platform(),
    "package_versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "outputs": sorted(
        str(path.relative_to(OUT_DIR))
        for path in OUT_DIR.rglob("*")
        if path.is_file()
    ),
}
(OUT_DIR / "notebook_07_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
display(manifest)